In [22]:
import pandas as dp 
import numpy as np
import matplotlib.pyplot as plt

Olharemos Homicidios Dolosos

1. Saber se existe BPM com muitos homicidios dolosos
2. Verificar se há uma media de hom dolosos dos BPMs
3. Fazer uma visualização dos que estão acima dos demais
4. Caso não haja muito acima dos demais, mostrar ranking de todos os BPMs
5. Mostre métricas de que sustentem que a média é confiável

In [23]:
try:
    df_ocorrencias = dp.read_csv('03.BaseDPEvolucaoMensalCisp.csv', sep=';', encoding='iso-8859-1')
    df_bpm = dp.read_csv('08.BPM.csv', sep=';', encoding='utf-8')

    print(df_ocorrencias.head())
    print(df_bpm.info())
    df_merge = df_ocorrencias.merge(df_bpm, left_on='aisp', right_on='COD_BPM', how='left') 
    df_doloso = df_merge[['aisp', 'NM_BPM', 'hom_doloso']]
    df_doloso.fillna('BPM não encontrada', inplace=True)
    df_doloso_agrupado = df_doloso.groupby(['aisp', 'NM_BPM']).sum().reset_index()
    print(df_doloso_agrupado)

    # df_merge['COD_BPM'] = df_merge['COD_BPM'].astype(str)
    
    # df_merge.fillna('BPM não encontrada', inplace=True)
    # df_ocorrencias.info()

except Exception as e:
    print(f"Erro: {e}")

   cisp  mes   ano  mes_ano  aisp  risp           munic    mcirc   regiao  \
0     1    1  2003  2003m01     5     1  Rio de Janeiro  3304557  Capital   
1     4    1  2003  2003m01     5     1  Rio de Janeiro  3304557  Capital   
2     5    1  2003  2003m01     5     1  Rio de Janeiro  3304557  Capital   
3     6    1  2003  2003m01     1     1  Rio de Janeiro  3304557  Capital   
4     7    1  2003  2003m01     1     1  Rio de Janeiro  3304557  Capital   

   hom_doloso  ...  cmp  cmba  ameaca  pessoas_desaparecidas  \
0           0  ...  NaN   NaN      21                      2   
1           3  ...  NaN   NaN      15                      6   
2           3  ...  NaN   NaN      47                      2   
3           6  ...  NaN   NaN      26                      2   
4           4  ...  NaN   NaN      10                      1   

   encontro_cadaver  encontro_ossada  pol_militares_mortos_serv  \
0                 0                0                          0   
1                 

In [24]:
doloso_array = np.array(df_doloso_agrupado['hom_doloso'])

q1 = np.percentile(doloso_array, 25)
q2 = np.percentile(doloso_array, 50)
q3 = np.percentile(doloso_array, 75)

media = np.mean(doloso_array)
desvio_padrao = np.std(doloso_array)
cv = desvio_padrao / media
delta = media/q2

In [27]:
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

df_doloso_outliers = df_doloso_agrupado[(df_doloso_agrupado['hom_doloso'] > limite_superior)]

df_grafico = None 
if df_doloso_outliers.empty:
    df_grafico = df_doloso_agrupado.sort_values(by='hom_doloso', ascending=False)
    print(f'Não há BPMs com nuúmero de homícidios dolosos muito maior que os outros)')
else:
    df_grafico = df_doloso_outliers
    print(f'1. Existem {df_doloso_outliers.shape[0]} BPMs com número de homicídios dolosos muito maior que os outros:')



1. Existem 2 BPMs com número de homicídios dolosos muito maior que os outros:


In [28]:
print(f'2. Média: {media}')
print(f'3. Mediana: {q2}')
print(f'4. Desvio Padrão: {desvio_padrao}')
print(f'5. Coeficiente de Variação: {cv*100:.2f}%')
print(f'6. Delta Media/Mediana: {delta:.2f}')

2. Média: 2496.095238095238
3. Mediana: 2175.5
4. Desvio Padrão: 2373.294965007448
5. Coeficiente de Variação: 95.08%
6. Delta Media/Mediana: 1.15
